# Enron Email Dataset — Topic Modelling Project (Email 5)

This notebook is an improved version of the previous pipeline.  
It **keeps the same project logic and structure**, but strengthens a few weak points:

- more careful normalization of noisy email artifacts,
- safer and better documented duplicate handling,
- slightly stronger stopword strategy for residual conversational noise,
- an explicit model-selection block with **perplexity + coherence + topic diversity**,
- a cleaner final interpretation section.

The goal is still the same: **prepare the Enron email corpus and identify the main themes of communication using topic modelling**.


In [1]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import kagglehub

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

pd.set_option("display.max_colwidth", 200)
RANDOM_STATE = 42


/Users/vasya/keras_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the dataset

The original Enron corpus contains raw email strings, including technical headers, forwarded chains, signatures, and other noise.  
The first task is to load the data and confirm the basic structure.


In [2]:
dataset_path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
csv_path = os.path.join(dataset_path, "emails.csv")

df = pd.read_csv(csv_path)

print("Dataset folder:", dataset_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing messages:", df["message"].isna().sum())
print("Duplicate rows:", df.duplicated().sum())

df.head()


Dataset folder: /Users/vasya/.cache/kagglehub/datasets/wcukierski/enron-email-dataset/versions/2
Shape: (517401, 2)
Columns: ['file', 'message']
Missing messages: 0
Duplicate rows: 0


,file,message
0,allen-p/_sent_mail/1.,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nConte..."
1,allen-p/_sent_mail/10.,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\n..."
2,allen-p/_sent_mail/100.,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: ..."
3,allen-p/_sent_mail/1000.,"Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>\nDate: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: randall.gay@enron.com\nSubject: \nMime-Version: 1.0\nCont..."
4,allen-p/_sent_mail/1001.,"Message-ID: <30922949.1075863688243.JavaMail.evans@thyme>\nDate: Thu, 31 Aug 2000 05:07:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: greg.piper@enron.com\nSubject: Re: Hello\nMime-Version: 1..."


## 2. Text-cleaning functions

The raw Enron emails are not ready for topic modelling.  
I apply the following sequence:

- remove technical headers  
- keep only the main message body  
- remove signatures, disclaimers, and contact noise  
- normalize text for modelling  
- flag probable newsletters / administrative emails


In [3]:
def remove_top_email_header(text):
    """Remove the technical top header from the raw Enron email."""
    if pd.isna(text):
        return ""

    text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text:
        return ""

    m = re.search(r'(?im)^X-FileName:.*$\n?', text)
    if m:
        return text[m.end():].strip()

    lines = text.split("\n")
    header_prefixes = (
        "Message-ID:", "Date:", "From:", "To:", "Subject:", "Mime-Version:",
        "Content-Type:", "Content-Transfer-Encoding:", "X-From:", "X-To:",
        "X-cc:", "X-bcc:", "X-Folder:", "X-Origin:", "X-FileName:"
    )

    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "":
            body_start = i + 1
            break

        if stripped.startswith(header_prefixes):
            continue

        body_start = i
        break

    return "\n".join(lines[body_start:]).strip()


def keep_only_main_message(text):
    """Keep the main content of the email and remove reply-chain wrappers."""
    if pd.isna(text):
        return ""

    text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text:
        return ""

    is_forwarded = bool(re.search(r'(?im)forwarded by .* on \d{1,2}/\d{1,2}/\d{2,4}', text))

    if is_forwarded:
        lines = text.split("\n")
        cleaned_lines = []
        skip_metadata = True
        started_body = False

        for line in lines:
            stripped = line.strip()

            if re.search(r'(?i)forwarded by .* on \d{1,2}/\d{1,2}/\d{2,4}', stripped):
                continue
            if re.match(r'^\d{1,2}:\d{2}\s*(AM|PM)\s*-*$', stripped, flags=re.I):
                continue

            if skip_metadata and (
                re.match(r'(?i)^from:\s+', stripped) or
                re.match(r'(?i)^to:\s+', stripped) or
                re.match(r'(?i)^cc:\s*', stripped) or
                re.match(r'(?i)^bcc:\s*', stripped) or
                re.match(r'(?i)^subject:\s+', stripped) or
                re.match(r'(?i)^sent:\s+', stripped) or
                re.match(r'(?i)^date:\s+', stripped) or
                re.match(r'(?i)^".*?"\s*<[^>]+>.*$', stripped) or
                re.match(r'^[A-Z][a-z]+\s+[A-Z][a-z]+.*\d{1,2}/\d{1,2}/\d{2,4}', stripped) or
                re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}.*$', stripped)
            ):
                continue

            if skip_metadata and stripped == "":
                continue

            if skip_metadata and stripped != "":
                skip_metadata = False
                started_body = True

            if started_body:
                cleaned_lines.append(line)

        text = "\n".join(cleaned_lines).strip()
        text = re.sub(r'(?m)^\s*>\s?', '', text)

    else:
        split_patterns = [
            r'(?im)^-+\s*original message\s*-+\s*$',
            r'(?im)^begin forwarded message:?\s*$',
            r'(?im)^on\s+.+wrote:\s*$',
            r'(?im)^-+\s*forwarded by\s+.*$'
        ]

        cut_positions = []
        for pattern in split_patterns:
            m = re.search(pattern, text)
            if m:
                cut_positions.append(m.start())

        if cut_positions:
            text = text[:min(cut_positions)].strip()

        text = re.sub(r'(?m)^\s*>\s?.*$', '', text)

    text = re.sub(r'(?im)^(to:|cc:|bcc:|subject:|from:|sent:|date:).*$\n?', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


def remove_signature_and_noise(text):
    """Remove signatures, disclaimers, contacts, URLs, HTML remnants, and footer noise."""
    if pd.isna(text):
        return ""

    text = str(text).strip()
    if not text:
        return ""

    text = re.sub(r'(?im)<embedded[^>]*>', ' ', text)
    text = re.sub(r'(?im)<[^>]+>', ' ', text)

    disclaimer_patterns = [
        r'(?im)^this message may contain confidential.*$',
        r'(?im)^the information contained in this e-mail.*$',
        r'(?im)^internet communications are not secure.*$',
        r'(?im)^please consider the environment before printing.*$',
        r'(?im)^this e-mail and any attachments are confidential.*$',
        r'(?im)^if you are not the intended recipient.*$'
    ]

    cut_positions = []
    for pattern in disclaimer_patterns:
        m = re.search(pattern, text)
        if m:
            cut_positions.append(m.start())

    if cut_positions:
        text = text[:min(cut_positions)].strip()

    text = re.sub(r'(?im)^(phone|fax|cell|mobile|pager|tel|telephone):.*$', ' ', text)
    text = re.sub(r'(?im)^.*\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b.*$', ' ', text)
    text = re.sub(r'(?im)^.*@\S+.*$', ' ', text)
    text = re.sub(r'(?im)^http\S+.*$', ' ', text)
    text = re.sub(r'(?im)^www\.\S+.*$', ' ', text)
    text = re.sub(r'(?m)^[-_=]{2,}\s*$', ' ', text)
    text = re.sub(r'(?im)^(attachment|attachments):.*$', ' ', text)
    text = re.sub(r'(?im)^inline attachment follows.*$', ' ', text)

    signature_markers = [
        r'(?im)^thanks[,]?\s*$',
        r'(?im)^thank you[,]?\s*$',
        r'(?im)^regards[,]?\s*$',
        r'(?im)^best[,]?\s*$',
        r'(?im)^best regards[,]?\s*$',
        r'(?im)^sincerely[,]?\s*$',
        r'(?im)^cheers[,]?\s*$'
    ]

    sig_positions = []
    for pattern in signature_markers:
        m = re.search(pattern, text)
        if m:
            sig_positions.append(m.start())

    if sig_positions:
        text = text[:min(sig_positions)].strip()

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)

    return text.strip()


def normalize_for_topic_model(text):
    """Normalize text for topic modelling and suppress residual technical artifacts."""
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)

    # remove common email / HTML / encoding remnants
    text = re.sub(r'\b(?:font|html|width|height|align|border|cellpadding|cellspacing|gif|jpeg|png|nbsp)\b', ' ', text)
    text = re.sub(r'\b(?:forwarded|original message|inline attachment follows|attachment|attachments)\b', ' ', text)
    text = re.sub(r'\b(?:pm|am|mon|tue|wed|thu|thur|fri|sat|sun)\b', ' ', text)

    # keep alphabetic tokens only: years / IDs / codes usually add noise here
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\b[a-z]\b', ' ', text)
    text = re.sub(r'\b(?:image|images|id|type|final|vince|database|program)\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def is_probably_newsletter(text):
    if pd.isna(text):
        return False

    t = str(text).lower()
    indicators = 0

    if len(t.split()) > 1000:
        indicators += 1
    if t.count("http") > 3:
        indicators += 1
    if "unsubscribe" in t:
        indicators += 1
    if "click here" in t:
        indicators += 1
    if "newsletter" in t:
        indicators += 1
    if "view this email" in t:
        indicators += 1

    return indicators >= 2


def is_admin_or_survey(text):
    if pd.isna(text):
        return False

    t = str(text).lower()
    keywords = [
        "full name",
        "login id",
        "office location",
        "distribution groups",
        "shared calendar",
        "email calendar",
        "what type of computer do you have",
        "do you have a pda",
        "normal work hours",
        "survey",
        "questionnaire",
        "please complete"
    ]

    return sum(kw in t for kw in keywords) >= 2


## 3. Quick audit of the cleaning pipeline

I inspect a small sample of emails to verify that the cleaning functions remove headers and obvious noise without destroying the core message.


In [4]:
sample_idx = [1, 9, 15, 20, 50, 100, 200]

for idx in sample_idx:
    print("=" * 100)
    print(f"INDEX: {idx}")
    print("=" * 100)

    original = df.loc[idx, "message"]
    body = remove_top_email_header(original)
    main = keep_only_main_message(body)
    clean = remove_signature_and_noise(main)

    print("ORIGINAL:")
    print(original[:1200])

    print("\n" + "-" * 100)
    print("CLEANED:")
    print(clean[:1200])
    print("\n")


INDEX: 1
ORIGINAL:
Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>
Date: Fri, 4 May 2001 13:51:00 -0700 (PDT)
From: phillip.allen@enron.com
To: john.lavorato@enron.com
Subject: Re:
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare a presentation.  I would suggest holding the business plan meetings here then take a trip without any formal business meetings.  I would even try and get some honest opinions on whether a trip is even desired or necessary.

As far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is 

## 4. Build the cleaned corpus

This stage keeps the original logic, but makes the diagnostic output a bit stronger.
We build several intermediate text versions and keep them for audit:

- `body_text` – after technical header removal,
- `main_message` – after reply-chain trimming,
- `clean_message` – after signatures / footers / obvious noise removal,
- `model_text` – normalized text used for topic modelling.

The goal is to keep the pipeline transparent and debuggable.


In [5]:
df["body_text"] = df["message"].apply(remove_top_email_header)
df["main_message"] = df["body_text"].apply(keep_only_main_message)
df["clean_message"] = df["main_message"].apply(remove_signature_and_noise)
df["model_text"] = df["clean_message"].apply(normalize_for_topic_model)

df["body_word_len"] = df["body_text"].str.split().str.len()
df["main_word_len"] = df["main_message"].str.split().str.len()
df["clean_word_len"] = df["clean_message"].str.split().str.len()
df["model_word_len"] = df["model_text"].str.split().str.len()

df["newsletter_flag"] = df["clean_message"].apply(is_probably_newsletter)
df["admin_flag"] = df["clean_message"].apply(is_admin_or_survey)

print("FINAL DATASET READY")
print("Rows in original df:", len(df))
print()
print(df["model_word_len"].describe())
print()
print("Newsletter-like rows:", int(df["newsletter_flag"].sum()))
print("Admin / survey rows:", int(df["admin_flag"].sum()))


FINAL DATASET READY
Rows in original df: 517401

count    517401.000000
mean        182.193944
std         788.127168
min           0.000000
25%          23.000000
50%          63.000000
75%         156.000000
max      136311.000000
Name: model_word_len, dtype: float64

Newsletter-like rows: 11793
Admin / survey rows: 474


### Filtering decision

The original project removed short texts, newsletters, and admin / survey content.  
This improved version keeps the same logic, but makes duplicate handling more explicit:

- filtering is done **before modelling**,
- duplicate removal is still **exact-text deduplication** on `model_text`,
- we also report how large the duplicate block is, because this matters for methodological transparency.


In [6]:
df_model = df[
    (df["model_word_len"] >= 10) &
    (df["model_word_len"] <= 1000) &
    (~df["newsletter_flag"]) &
    (~df["admin_flag"])
].copy()

before_n = len(df_model)

duplicate_counts = (
    df_model["model_text"]
    .value_counts(dropna=False)
    .rename_axis("model_text")
    .reset_index(name="count")
)

repeated_templates = duplicate_counts[duplicate_counts["count"] > 1].sort_values("count", ascending=False)

df_model = df_model.drop_duplicates(subset=["model_text"]).copy()
after_n = len(df_model)

print("Before duplicate removal:", before_n)
print("After duplicate removal:", after_n)
print("Removed exact duplicates:", before_n - after_n)
print("Duplicate share (%):", round((before_n - after_n) / before_n * 100, 2))

print("\nTop repeated cleaned texts (for audit):")
repeated_templates.head(10)


Before duplicate removal: 441190
After duplicate removal: 191200
Removed exact duplicates: 249990
Duplicate share (%): 56.66

Top repeated cleaned texts (for audit):


,model_text,count
0,start date hourahead hour no ancillary schedules awarded no variances detected log messages,1700
1,start date hourahead hour no ancillary schedules awarded no variances detected log messages parsing file portland westdesk california scheduling iso,1374
2,start date hourahead hour hourahead schedule download failed manual intervention required,894
3,the report named east totals published as of is now available for viewing on the website,264
4,start date hourahead hour no ancillary schedules awarded no variances detected log messages error retrieving hourahead price data process continuing,257
5,start date hourahead hour hourahead schedule download failed manual intervention required log messages error dbcaps data cannot perform this operation on closed unknown alias dbcaps data unknown a...,249
6,this warning is sent automatically to inform you that your mailbox is approaching the maximum size limit your mailbox size is currently kb mailbox size limits when your mailbox reaches kb you will...,232
7,following please find the daily enrononline executive summary enrononline executive summary for transaction summary external transactions today average daily external transactions day trailing avg...,230
8,task assignment status completed task priority task due on task start date,194
9,get your free download of msn explorer at report xls,161


## 5. Prepare a stronger modelling subset

We keep the same modelling idea, but make the subset slightly stricter:

- only reasonably substantial emails are used,
- extremely short emails are excluded because they often contribute generic conversational noise,
- the corpus is reset cleanly before vectorization.


In [7]:
df_topics_input = df_model[
    (df_model["model_word_len"] >= 40) &
    (df_model["model_word_len"] <= 800)
].copy()

docs = df_topics_input["model_text"].dropna().astype(str)
docs = docs[docs.str.strip() != ""].reset_index(drop=True)

print("Number of documents used for modelling:", len(docs))
docs.head()


Number of documents used for modelling: 130058


0    traveling to have business meeting takes the fun out of the trip especially if you have to prepare presentation would suggest holding the business plan meetings here then take trip without any for...
1    phillip as discussed during our phone conversation in parallon microturbine power generation deal for national accounts customer developing proposal to sell power to customer at fixed or collar fl...
2    lucy here are the rentrolls open them and save in the rentroll folder follow these steps so you don misplace these files click on save as click on the drop down triangle under save in click on the...
3    please respond to westgate enclosed are demographics on the westgate site from investor alliance investor alliance says that these demographics are similar to the package on san marcos that you re...
4    liane as we discussed yesterday concerned there may have been an attempt to manipulate the el paso san juan monthly index it appears that single buyer entered the marketplace 

In [23]:
from collections import Counter
import pandas as pd

def frequent_terms_by_length(series, min_len=1, max_len=3, top_n=50):
    tokens = " ".join(series.dropna()).split()
    counts = Counter(
        token for token in tokens
        if min_len <= len(token) <= max_len
    )
    return (
        pd.DataFrame(counts.items(), columns=["term", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
        .head(top_n)
    )

print("Most frequent terms with length 1–3:")
display(frequent_terms_by_length(df["clean_message"], min_len=1, max_len=3, top_n=50))

print("Most frequent terms with length 4:")
display(frequent_terms_by_length(df["clean_message"], min_len=4, max_len=4, top_n=50))

print("Most frequent terms with length 5:")
display(frequent_terms_by_length(df["clean_message"], min_len=5, max_len=5, top_n=50))

Most frequent terms with length 1–3:


,term,count
0,the,4124837
1,to,2845947
2,and,2012339
3,of,1906952
4,a,1463990
5,in,1310789
6,for,1097876
7,is,947253
8,on,827577
9,you,746620


Most frequent terms with length 4:


,term,count
0,that,846929
1,will,601764
2,with,597732
3,this,549572
4,have,549152
5,from,406968
6,your,351218
7,been,158782
8,said,155927
9,they,151589


Most frequent terms with length 5:


,term,count
0,Enron,321199
1,would,261154
2,about,167946
3,which,167176
4,power,158455
5,their,148015
6,other,122870
7,these,100962
8,could,95626
9,there,92701


## 6. Shared helpers for modelling

This is still the same modelling block, but with two improvements:

1. a slightly stronger stopword set for residual conversational and technical noise;
2. an additional **topic diversity** metric, so model choice is not based on perplexity alone.


In [25]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = set(ENGLISH_STOP_WORDS).union({
    # company / email metadata noise
    "enron", "ect", "hou", "corp", "com", "mail", "email", "message",
    "original", "attached", "attachment", "subject", "cc", "bcc",
    "fwd", "fw", "re", "forwarded",

    # common email politeness / conversational noise
    "thanks", "thank", "please", "let", "know", "need",
    "pm", "am", "ll", "okay", "ok", "yes", "no",

    # relative-time conversational noise
    "today", "tomorrow", "yesterday",

    # generic conversational verbs
    "said", "going", "want", "make", "made", "getting",
    "got", "sent", "send",

    # frequent person-name noise
    "phillip", "allen", "john", "randy", "david", "dave", "mike",
    "ina", "julie", "paula", "brenda", "cynthia",

    # weak topic words that were repeatedly noisy
    "question", "questions"
})

extra_stopwords = {
    # weak conversational / stylistic noise
    "did", "really", "right", "way", "little", "come", "look",
    "home", "night", "sure",

    # generic document / admin noise
    "following", "comments", "file", "doc", "copy", "draft", "list",

    # short-token / contraction artifacts
    "don", "ve", "th", "eb"
}

disclaimer_stopwords = {
    # disclaimer / html / web artifacts
    "click", "intended", "recipient", "web", "site",
    "link", "links", "error", "receive",
    "font", "size", "iso", "trans", "td", "http",
    "www", "html", "width", "height", "align", "border",
    "cellpadding", "cellspacing", "color", "gif", "images",
    "image", "id", "type"
}

final_stopwords = sorted(
    custom_stopwords
    .union(extra_stopwords)
    .union(disclaimer_stopwords)
)


def build_count_vectorizer():
    return CountVectorizer(
        stop_words=final_stopwords,
        max_df=0.35,
        min_df=20,
        ngram_range=(1, 2),
        max_features=5000,
        token_pattern=r'(?u)\b[a-z][a-z]+\b'
    )


def build_tfidf_vectorizer():
    return TfidfVectorizer(
        stop_words=final_stopwords,
        max_df=0.35,
        min_df=20,
        ngram_range=(1, 2),
        max_features=5000,
        token_pattern=r'(?u)\b[a-z][a-z]+\b'
    )


def get_topic_words(model, feature_names, n_top_words=15):
    topic_word_lists = []
    for topic in model.components_:
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topic_word_lists.append(top_words)
    return topic_word_lists


def print_topics(model, feature_names, n_top_words=15):
    topic_word_lists = get_topic_words(model, feature_names, n_top_words=n_top_words)
    for topic_idx, words in enumerate(topic_word_lists):
        print(f"TOPIC {topic_idx}:")
        print(", ".join(words))
        print("-" * 120)


def compute_topic_diversity(model, feature_names, top_n=15):
    """Share of unique words among all top topic words."""
    topic_word_lists = get_topic_words(model, feature_names, n_top_words=top_n)
    flat_words = [word for topic in topic_word_lists for word in topic]
    return len(set(flat_words)) / len(flat_words)


def summarize_top_terms(doc_series, n=30):
    counts = Counter(" ".join(doc_series).split())
    return pd.DataFrame(counts.most_common(n), columns=["term", "count"])


## 7. Baseline LDA model

We keep the simple baseline LDA, but now it uses the strengthened vectorizer settings from the helper block.


In [26]:
baseline_vectorizer = build_count_vectorizer()
baseline_dtm = baseline_vectorizer.fit_transform(docs)
baseline_feature_names = baseline_vectorizer.get_feature_names_out()

lda_baseline = LatentDirichletAllocation(
    n_components=3,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=30
)

lda_baseline.fit(baseline_dtm)

print("Baseline document-term matrix shape:", baseline_dtm.shape)
print_topics(lda_baseline, baseline_feature_names, n_top_words=20)


Baseline document-term matrix shape: (130058, 5000)
TOPIC 0:
energy, california, company, state, ferc, power, information, confidential, employees, commission, utility, million, consumers, use, electricity, utilities, bankruptcy, received, new, stock
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
time, like, just, meeting, houston, new, week, good, think, work, day, free, information, group, people, office, great, friday, hope, year
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
gas, agreement, market, power, deal, new, price, information, trading, energy, credit, time, contract, ena, day, review, report, business, changes, company
------------------------------------------------------------------------------------------------------------------------


## 8. Compare candidate LDA models

Instead of relying only on perplexity, this block compares candidate models on:

- **perplexity**,
- **topic diversity** (less overlap between topics is better),
- qualitative interpretability from the printed topic words.


In [27]:
vectorizer_compare = build_count_vectorizer()
dtm_compare = vectorizer_compare.fit_transform(docs)
feature_names_compare = vectorizer_compare.get_feature_names_out()

results = []
lda_models = {}
topic_options = [3, 5, 7, 10]

for n_topics in topic_options:
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE,
        learning_method="batch",
        max_iter=40
    )

    lda_model.fit(dtm_compare)
    lda_models[n_topics] = lda_model

    perplexity_value = lda_model.perplexity(dtm_compare)
    topic_diversity_value = compute_topic_diversity(lda_model, feature_names_compare, top_n=15)

    results.append({
        "n_topics": n_topics,
        "perplexity": perplexity_value,
        "topic_diversity": round(topic_diversity_value, 4)
    })

    print("=" * 120)
    print(f"LDA WITH {n_topics} TOPICS")
    print("=" * 120)
    print(f"Perplexity: {perplexity_value:.2f}")
    print(f"Topic diversity: {topic_diversity_value:.4f}\n")
    print_topics(lda_model, feature_names_compare, n_top_words=15)
    print("\n")

results_df = pd.DataFrame(results).sort_values(["perplexity", "topic_diversity"], ascending=[True, False]).reset_index(drop=True)
results_df


LDA WITH 3 TOPICS
Perplexity: 2205.42
Topic diversity: 0.8667

TOPIC 0:
energy, california, company, state, power, confidential, information, ferc, employees, commission, utility, million, consumers, electricity, use
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
time, like, meeting, just, houston, new, week, good, think, work, day, information, free, group, people
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
gas, agreement, market, power, deal, new, price, information, trading, credit, contract, time, energy, ena, review
------------------------------------------------------------------------------------------------------------------------


LDA WITH 5 TOPICS
Perplexity: 1964.48
Topic diversity: 0.8133

TOPIC 0:
employees, confidential, company, information, energy, california, folder, privileged, sender, consumers, fut

,n_topics,perplexity,topic_diversity
0,10,1722.038862,0.8067
1,7,1874.872213,0.8190
2,5,1964.483060,0.8133
3,3,2205.415957,0.8667


## 9. NMF as a comparison model

NMF is kept as an alternative topic model so the analysis is not dependent on a single method.


In [28]:
tfidf_vectorizer = build_tfidf_vectorizer()
tfidf = tfidf_vectorizer.fit_transform(docs)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

nmf_model = NMF(
    n_components=7,
    random_state=RANDOM_STATE,
    init="nndsvda",
    max_iter=400
)

W = nmf_model.fit_transform(tfidf)

print("=" * 120)
print("NMF WITH 7 TOPICS")
print("=" * 120)
print_topics(nmf_model, tfidf_feature_names, n_top_words=15)

dominant_topics_nmf = np.argmax(W, axis=1)
topic_distribution_nmf = pd.Series(dominant_topics_nmf).value_counts().sort_index()

topic_distribution_nmf_df = pd.DataFrame({
    "topic": topic_distribution_nmf.index,
    "document_count": topic_distribution_nmf.values,
    "share_percent": (topic_distribution_nmf.values / len(docs) * 100).round(2)
})

topic_distribution_nmf_df


NMF WITH 7 TOPICS
TOPIC 0:
power, energy, market, gas, new, trading, california, business, company, ferc, information, services, risk, group, order
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
employees, consumers, company, california, company declared, declared bankruptcy, company employees, energy bills, donate, declared, retirement, millions, bills, energy, funds
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
agreement, sara, north america, america, north, smith, houston, smith street, isda, houston texas, street houston, street, texas, master, america smith
------------------------------------------------------------------------------------------------------------------------
TOPIC 3:
time, meeting, just, like, good, think, week, work, hope, day, great, friday, talk, morning, people
----------------------------------

,topic,document_count,share_percent
0,0,32341,24.87
1,1,1650,1.27
2,2,15202,11.69
3,3,52354,40.25
4,4,9005,6.92
5,5,3822,2.94
6,6,15684,12.06


## 10. Remove obvious residual noise

This is still a light post-cleaning step, not a full new pipeline.
The purpose is only to remove clearly non-substantive residual texts that survived the earlier rules.


In [29]:
noise_patterns = [
    "web specials",
    "thank you for subscribing",
    "inline attachment follows",
    "outlook migration",
    "calendar entry",
    "view this email",
    "unsubscribe",
    "distribution groups",
    "shared calendar",
    "please complete",
    "questionnaire"
]

mask_keep = ~docs.str.contains("|".join(noise_patterns), case=False, na=False, regex=True)
docs_denoised = docs[mask_keep].reset_index(drop=True)

print("Original number of documents:", len(docs))
print("Remaining after removing obvious noise:", len(docs_denoised))
print("Removed documents:", len(docs) - len(docs_denoised))
print("Share removed (%):", round((len(docs) - len(docs_denoised)) / len(docs) * 100, 2))
print()
print("Most frequent terms after denoising audit:")
summarize_top_terms(docs_denoised, n=25)


Original number of documents: 130058
Remaining after removing obvious noise: 127207
Removed documents: 2851
Share removed (%): 2.19

Most frequent terms after denoising audit:


,term,count
0,the,999316
1,to,691412
2,and,460917
3,of,391877
4,you,300734
5,in,292802
6,for,271675
7,is,243269
8,that,217240
9,on,203981


## 11. Final LDA model on the denoised corpus

The final model keeps the original project idea of using **7 topics**, but now on a slightly better cleaned corpus and with stronger modelling settings.

Why still 7 topics?
- it usually gives a better balance than 3 or 5,
- it is often more interpretable than 10 when corporate email topics start to fragment,
- it remains easy to explain in a final report.


In [30]:
vectorizer_final = build_count_vectorizer()
dtm_final = vectorizer_final.fit_transform(docs_denoised)
feature_names_final = vectorizer_final.get_feature_names_out()

lda_final = LatentDirichletAllocation(
    n_components=7,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=40
)

doc_topic_matrix_final = lda_final.fit_transform(dtm_final)

print("=" * 120)
print("FINAL LDA: 7 TOPICS ON DENOISED CORPUS")
print("=" * 120)
print(f"Perplexity: {lda_final.perplexity(dtm_final):.2f}")
print(f"Topic diversity: {compute_topic_diversity(lda_final, feature_names_final, top_n=15):.4f}\n")
print_topics(lda_final, feature_names_final, n_top_words=15)

dominant_topics_final = np.argmax(doc_topic_matrix_final, axis=1)
topic_distribution_final = pd.Series(dominant_topics_final).value_counts().sort_index()

topic_distribution_final_df = pd.DataFrame({
    "topic": topic_distribution_final.index,
    "document_count": topic_distribution_final.values,
    "share_percent": (topic_distribution_final.values / len(docs_denoised) * 100).round(2)
})

topic_distribution_final_df


FINAL LDA: 7 TOPICS ON DENOISED CORPUS
Perplexity: 1838.50
Topic diversity: 0.8381

TOPIC 0:
just, think, like, good, time, work, hope, day, great, week, people, things, new, talk, weekend
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
power, market, energy, business, new, risk, company, issues, ferc, group, management, information, trading, year, services
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
new, information, free, received, service, internet, available, area, address, online, order, access, futures, content, price
------------------------------------------------------------------------------------------------------------------------
TOPIC 3:
agreement, credit, legal, information, review, master, ena, letter, sara, america, north, north america, received, contract, isda
------------------------------------------

,topic,document_count,share_percent
0,0,21478,16.88
1,1,23762,18.68
2,2,8871,6.97
3,3,25997,20.44
4,4,1651,1.30
5,5,24523,19.28
6,6,20925,16.45


## 12. Representative emails for each final topic

Top words are useful, but they are not enough on their own.  
To interpret each topic more reliably, I inspect the highest-scoring representative emails.


In [31]:
topic_word_lists_final = get_topic_words(lda_final, feature_names_final, n_top_words=12)

n_examples = 3
for topic_idx in range(7):
    print("=" * 140)
    print(f"TOPIC {topic_idx}")
    print("=" * 140)
    print("TOP WORDS:", ", ".join(topic_word_lists_final[topic_idx]))
    print()

    top_doc_indices = np.argsort(doc_topic_matrix_final[:, topic_idx])[::-1][:n_examples]

    for rank, doc_idx in enumerate(top_doc_indices, start=1):
        topic_score = doc_topic_matrix_final[doc_idx, topic_idx]
        print(f"[Representative email #{rank}]  document_position={doc_idx}  topic_score={topic_score:.4f}")
        print(docs_denoised.iloc[doc_idx][:2500])
        print("\n" + "-" * 140 + "\n")


TOPIC 0
TOP WORDS: just, think, like, good, time, work, hope, day, great, week, people, things

[Representative email #1]  document_position=23833  topic_score=0.9978
start date hourahead hour no ancillary schedules awarded variances detected variances detected in energy import export schedule log messages error retrieving hourahead price data process continuing energy import export schedule hour bad data from iso trans sc ectstsw mkt trans date tie point pverde devers interchg ciso epmi engy nfrm hour bad data from iso trans sc ectstsw mkt trans date tie point fcornr psuedo interchg epmi ciso engy firm hour bad data from iso trans sc ectstsw mkt trans date tie point mead walc interchg epmi ciso engy firm hour bad data from iso trans sc ectrt mkt trans date tie point malin rndmtn interchg epmi ciso aaa engy wheel hour bad data from iso trans sc ectrt mkt trans date tie point fcornr psuedo interchg epmi ciso aaa engy wheel hour bad data from iso trans sc ectrt mkt trans date tie point p

## 13. Final model-selection diagnostics

This block makes the model choice more defensible by adding **coherence** on the denoised corpus.
Together with perplexity and topic diversity, this gives a more balanced basis for selecting the final solution.


In [32]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


def compute_lda_diagnostics(doc_series, n_topics):
    vectorizer = build_count_vectorizer()
    dtm = vectorizer.fit_transform(doc_series)
    feature_names = vectorizer.get_feature_names_out()

    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE,
        learning_method="batch",
        max_iter=40
    )
    lda_model.fit(dtm)

    topic_word_lists = get_topic_words(lda_model, feature_names, n_top_words=15)
    tokenized_docs = [doc.split() for doc in doc_series]
    dictionary = Dictionary(tokenized_docs)

    coherence_model = CoherenceModel(
        topics=topic_word_lists,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )

    return {
        "n_topics": n_topics,
        "perplexity": round(lda_model.perplexity(dtm), 2),
        "coherence_c_v": round(coherence_model.get_coherence(), 4),
        "topic_diversity": round(compute_topic_diversity(lda_model, feature_names, top_n=15), 4)
    }


diagnostics_df = pd.DataFrame(
    [compute_lda_diagnostics(docs_denoised, n) for n in [3, 5, 7, 10]]
).sort_values(
    ["coherence_c_v", "topic_diversity", "perplexity"],
    ascending=[False, False, True]
).reset_index(drop=True)

diagnostics_df


,n_topics,perplexity,coherence_c_v,topic_diversity
0,10,1680.67,0.5844,0.8000
1,7,1838.50,0.5359,0.8381
2,5,1988.02,0.4855,0.8533
3,3,2201.50,0.4048,0.8667


## 14. Final interpretation notes

You can keep this section as the final written conclusion in your submission.

### Suggested final interpretation

This project used the Enron email corpus to identify the main themes of internal corporate communication through topic modelling.  
The pipeline included multi-stage cleaning, removal of technical headers, trimming of reply chains, suppression of signatures and disclaimer noise, filtering of newsletter-like and administrative messages, and final vectorization for LDA and NMF.

The final results suggest that the most stable and interpretable themes are related to:

- **energy trading and market operations**,  
- **gas and power markets**,  
- **contracts, credit, and risk-related communication**,  
- **internal coordination and operational planning**.

The improved version of the notebook also shows that model selection should not rely on one metric only.  
Perplexity, coherence, and topic diversity together provide a more defensible basis for choosing the final number of topics.  
Even after careful cleaning, some residual conversational noise remains, which is a normal limitation of large real-world email corpora.

### Honest limitation to mention

A remaining limitation is that corporate email language is highly repetitive and often contains semi-generic coordination messages.  
Because of that, some topics may still include overlap between operational communication and substantive business discussion.

### Final submission sentence you can use

Overall, the project produced a reasonably clean and interpretable topic structure, showing that the Enron email dataset is suitable for NLP-based thematic analysis, while also demonstrating the practical difficulty of fully eliminating noise in real corporate communication data.
